In [1]:
import pandas as pd
import numpy as np

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# Step 2 — Load feature dataset

feature_df = pd.read_csv("../dataset/feature_dataset.csv")

print("Feature dataset loaded successfully!")
print("Shape:", feature_df.shape)

display(feature_df.head(10))

Feature dataset loaded successfully!
Shape: (1008, 17)


,period,department,category,transaction_count,total_amount,budget,variance,budget_utilization_pct,variance_pct,budget_available,avg_transaction_value,spending_deviation_pct,transaction_count_deviation_pct,period_total_spending,period_spending_change_pct,rolling_3m_avg_spending,rolling_spending_deviation_pct
0,2025-01,Customer Support,Outsourcing,79,906916.46,913791.70,-6875.24,99.247614,-0.752386,1,11479.955190,0.903640,0.708215,53363242.28,NaN,53363242.28,0.0
1,2025-01,Customer Support,Product Sales,56,1273072.74,NaN,NaN,NaN,NaN,0,22733.441786,9.997398,11.258278,53363242.28,NaN,53363242.28,0.0
2,2025-01,Customer Support,Service Revenue,50,1125824.05,NaN,NaN,NaN,NaN,0,22516.481000,3.065599,6.257379,53363242.28,NaN,53363242.28,0.0
3,2025-01,Customer Support,Software,82,724787.64,737863.28,-13075.64,98.227905,-1.772095,1,8838.873659,2.270978,9.414381,53363242.28,NaN,53363242.28,0.0
4,2025-01,Customer Support,Subscription,60,1456901.64,NaN,NaN,NaN,NaN,0,24281.694000,34.795807,26.909518,53363242.28,NaN,53363242.28,0.0
5,2025-01,Customer Support,Support Tools,69,658831.34,640835.18,17996.16,102.808235,2.808235,1,9548.280290,-12.701382,-13.929314,53363242.28,NaN,53363242.28,0.0
6,2025-01,Customer Support,Training,77,483785.12,477200.44,6584.68,101.379856,1.379856,1,6282.923636,-11.456883,0.946832,53363242.28,NaN,53363242.28,0.0
7,2025-01,Customer Support,Transaction Fees,31,721542.13,NaN,NaN,NaN,NaN,0,23275.552581,-32.509749,-35.266821,53363242.28,NaN,53363242.28,0.0
8,2025-01,Finance,Audit,49,276168.70,270906.81,5261.89,101.942325,1.942325,1,5636.095918,-23.223205,-4.751620,53363242.28,NaN,53363242.28,0.0
9,2025-01,Finance,Bank Charges,62,278818.88,258334.99,20483.89,107.929197,7.929197,1,4497.078710,19.364467,20.910076,53363242.28,NaN,53363242.28,0.0


In [3]:
# Step 3 — Inspect feature dataset

print("Columns:")
print(feature_df.columns.tolist())

print("\nData types:")
print(feature_df.dtypes)

print("\nMissing values:")
print(feature_df.isna().sum())

Columns:
['period', 'department', 'category', 'transaction_count', 'total_amount', 'budget', 'variance', 'budget_utilization_pct', 'variance_pct', 'budget_available', 'avg_transaction_value', 'spending_deviation_pct', 'transaction_count_deviation_pct', 'period_total_spending', 'period_spending_change_pct', 'rolling_3m_avg_spending', 'rolling_spending_deviation_pct']

Data types:
period                              object
department                          object
category                            object
transaction_count                    int64
total_amount                       float64
budget                             float64
variance                           float64
budget_utilization_pct             float64
variance_pct                       float64
budget_available                     int64
avg_transaction_value              float64
spending_deviation_pct             float64
transaction_count_deviation_pct    float64
period_total_spending              float64
period_spending_

In [4]:
# Step 4 — Complete missing-value analysis

missing_summary = (
    feature_df.isna()
    .sum()
    .sort_values(ascending=False)
)

print("Missing-value summary:")
display(missing_summary.to_frame("missing_count"))

Missing-value summary:


,missing_count
variance_pct,504
variance,504
budget_utilization_pct,504
budget,504
period_spending_change_pct,56
period,0
total_amount,0
category,0
transaction_count,0
department,0


In [5]:
# Step 5 — Prepare anomaly-detection features

anomaly_df = feature_df.copy()

# Features available for every transaction group
base_anomaly_features = [
    "total_amount",
    "transaction_count",
    "avg_transaction_value",
    "spending_deviation_pct",
    "transaction_count_deviation_pct",
    "period_total_spending",
    "period_spending_change_pct",
    "rolling_3m_avg_spending",
    "rolling_spending_deviation_pct"
]

# Features available only when a budget exists
budget_anomaly_features = [
    "variance",
    "budget_utilization_pct",
    "variance_pct"
]

print("Base anomaly features:")
print(base_anomaly_features)

print("\nBudget-based anomaly features:")
print(budget_anomaly_features)

print("\nBudget availability:")
print(anomaly_df["budget_available"].value_counts())

Base anomaly features:
['total_amount', 'transaction_count', 'avg_transaction_value', 'spending_deviation_pct', 'transaction_count_deviation_pct', 'period_total_spending', 'period_spending_change_pct', 'rolling_3m_avg_spending', 'rolling_spending_deviation_pct']

Budget-based anomaly features:
['variance', 'budget_utilization_pct', 'variance_pct']

Budget availability:
budget_available
1    504
0    504
Name: count, dtype: int64


In [6]:
# Step 6 — Create anomaly signals

# Budget utilization anomaly
anomaly_df["high_budget_utilization"] = (
    anomaly_df["budget_available"].eq(1)
    & (anomaly_df["budget_utilization_pct"] > 100)
)

# Large spending deviation
anomaly_df["high_spending_deviation"] = (
    anomaly_df["spending_deviation_pct"].abs() > 20
)

# Large transaction-count deviation
anomaly_df["high_transaction_deviation"] = (
    anomaly_df["transaction_count_deviation_pct"].abs() > 20
)

# Large period-over-period spending change
anomaly_df["large_period_change"] = (
    anomaly_df["period_spending_change_pct"].abs() > 20
)

# Large deviation from 3-month rolling average
anomaly_df["large_rolling_deviation"] = (
    anomaly_df["rolling_spending_deviation_pct"].abs() > 20
)

print("Anomaly signals created.")

signal_columns = [
    "high_budget_utilization",
    "high_spending_deviation",
    "high_transaction_deviation",
    "large_period_change",
    "large_rolling_deviation"
]

display(anomaly_df[signal_columns].sum().to_frame("flagged_rows"))

Anomaly signals created.


,flagged_rows
high_budget_utilization,343
high_spending_deviation,249
high_transaction_deviation,173
large_period_change,56
large_rolling_deviation,0


In [7]:
# Step 7 — Calculate anomaly score

anomaly_df["anomaly_score"] = (
    anomaly_df["high_budget_utilization"].astype(int)
    + anomaly_df["high_spending_deviation"].astype(int)
    + anomaly_df["high_transaction_deviation"].astype(int)
    + anomaly_df["large_period_change"].astype(int)
    + anomaly_df["large_rolling_deviation"].astype(int)
)

# Create anomaly flag
anomaly_df["is_anomaly"] = (
    (anomaly_df["anomaly_score"] >= 2)
    |
    (
        (anomaly_df["high_spending_deviation"] == True)
        & (anomaly_df["spending_deviation_pct"] >= 50)
    )
).astype(int)

print("Anomaly score calculated.")

print("\nAnomaly score distribution:")
print(anomaly_df["anomaly_score"].value_counts().sort_index())

print("\nAnomaly flag distribution:")
print(anomaly_df["is_anomaly"].value_counts())

display(
    anomaly_df[
        [
            "period",
            "department",
            "category",
            "anomaly_score",
            "is_anomaly"
        ]
    ]
    .sort_values("anomaly_score", ascending=False)
    .head(20)
)

Anomaly score calculated.

Anomaly score distribution:
anomaly_score
0    420
1    398
2    148
3     41
4      1
Name: count, dtype: int64

Anomaly flag distribution:
is_anomaly
0    817
1    191
Name: count, dtype: int64


,period,department,category,anomaly_score,is_anomaly
130,2025-03,HR,Recruitment,4,1
960,2026-06,Finance,Audit,3,1
965,2026-06,Finance,Software,3,1
974,2026-06,HR,Training,3,1
74,2025-02,HR,Recruitment,3,1
16,2025-01,HR,Benefits,3,1
114,2025-03,Customer Support,Service Revenue,3,1
124,2025-03,Finance,Service Revenue,3,1
447,2025-08,Sales,Travel,3,1
131,2025-03,HR,Service Revenue,3,1


In [8]:
# Step 8 — Analyze anomaly signal combinations

signal_columns = [
    "high_budget_utilization",
    "high_spending_deviation",
    "high_transaction_deviation",
    "large_period_change",
    "large_rolling_deviation"
]

# Create a readable combination of triggered signals
anomaly_df["anomaly_signals"] = anomaly_df[signal_columns].apply(
    lambda row: ", ".join(
        col.replace("_", " ")
        for col in signal_columns
        if row[col] == 1
    ),
    axis=1
)

print("Anomaly signal combinations:")
print(
    anomaly_df.loc[
        anomaly_df["is_anomaly"] == 1,
        "anomaly_signals"
    ].value_counts()
)

print("\nAnomalies by score:")
print(
    anomaly_df.loc[
        anomaly_df["is_anomaly"] == 1,
        "anomaly_score"
    ].value_counts().sort_index()
)

display(
    anomaly_df.loc[
        anomaly_df["is_anomaly"] == 1,
        [
            "period",
            "department",
            "category",
            "anomaly_score",
            "anomaly_signals"
        ]
    ]
    .sort_values(
        ["anomaly_score", "period"],
        ascending=[False, True]
    )
    .head(20)
)

Anomaly signal combinations:
anomaly_signals
high spending deviation, high transaction deviation                                                  85
high budget utilization, high spending deviation                                                     33
high budget utilization, high spending deviation, high transaction deviation                         31
high budget utilization, large period change                                                         18
high spending deviation, high transaction deviation, large period change                              9
high spending deviation, large period change                                                          7
high budget utilization, high transaction deviation                                                   4
high budget utilization, high spending deviation, large period change                                 1
high budget utilization, high spending deviation, high transaction deviation, large period change     1
high transaction de

,period,department,category,anomaly_score,anomaly_signals
130,2025-03,HR,Recruitment,4,"high budget utilization, high spending deviati..."
16,2025-01,HR,Benefits,3,"high budget utilization, high spending deviati..."
74,2025-02,HR,Recruitment,3,"high budget utilization, high spending deviati..."
112,2025-03,Customer Support,Outsourcing,3,"high budget utilization, high spending deviati..."
114,2025-03,Customer Support,Service Revenue,3,"high spending deviation, high transaction devi..."
116,2025-03,Customer Support,Subscription,3,"high spending deviation, high transaction devi..."
119,2025-03,Customer Support,Transaction Fees,3,"high spending deviation, high transaction devi..."
124,2025-03,Finance,Service Revenue,3,"high spending deviation, high transaction devi..."
131,2025-03,HR,Service Revenue,3,"high spending deviation, high transaction devi..."
158,2025-03,Operations,Transaction Fees,3,"high spending deviation, high transaction devi..."


In [9]:
# Step 9 — Analyze anomalies by department and category

# Anomaly count by department
department_anomalies = (
    anomaly_df[anomaly_df["is_anomaly"] == 1]
    .groupby("department")
    .size()
    .sort_values(ascending=False)
)

print("Anomalies by department:")
print(department_anomalies)


# Anomaly count by category
category_anomalies = (
    anomaly_df[anomaly_df["is_anomaly"] == 1]
    .groupby("category")
    .size()
    .sort_values(ascending=False)
)

print("\nAnomalies by category:")
print(category_anomalies)


# Anomaly count by period
period_anomalies = (
    anomaly_df[anomaly_df["is_anomaly"] == 1]
    .groupby("period")
    .size()
    .sort_values(ascending=False)
)

print("\nAnomalies by period:")
print(period_anomalies)


# Highest-risk department-category combinations
dept_category_anomalies = (
    anomaly_df[anomaly_df["is_anomaly"] == 1]
    .groupby(["department", "category"])
    .agg(
        anomaly_count=("is_anomaly", "sum"),
        avg_anomaly_score=("anomaly_score", "mean"),
        max_anomaly_score=("anomaly_score", "max")
    )
    .sort_values(
        ["anomaly_count", "avg_anomaly_score"],
        ascending=False
    )
)

print("\nTop department-category anomaly combinations:")
display(dept_category_anomalies.head(20))

Anomalies by department:
department
Finance             42
HR                  30
IT                  29
Sales               28
Customer Support    27
Operations          19
Marketing           16
dtype: int64

Anomalies by category:
category
Transaction Fees         27
Software                 24
Subscription             22
Product Sales            20
Service Revenue          19
Professional Services     8
Outsourcing               7
Recruitment               7
Audit                     5
Travel                    5
Training                  5
Advertising               4
Logistics                 4
Bank Charges              4
Benefits                  4
Cybersecurity             3
Commissions               3
Maintenance               3
Hardware                  3
Support Tools             3
Utilities                 3
Content                   2
Client Entertainment      2
Supplies                  2
Events                    1
Cloud Services            1
dtype: int64

Anomalies by pe

anomaly_count  avg_anomaly_score  \
department       category                                                  
Finance          Professional Services              8           2.125000   
                 Transaction Fees                   8           2.000000   
HR               Recruitment                        7           2.571429   
Customer Support Outsourcing                        7           2.285714   
Sales            Software                           6           2.333333   
Finance          Subscription                       6           2.000000   
IT               Subscription                       6           2.000000   
Sales            Travel                             5           2.400000   
Finance          Audit                              5           2.200000   
HR               Service Revenue                    5           2.200000   
IT               Software                           5           2.200000   
Customer Support Software                           4           2.500000   
Finance          Bank Charges                       4           2.500000   
HR               Benefits                           4           2.500000   
                 Training                           4           2.500000   
Operations       Logistics                          4           2.500000   
Customer Support Service Revenue                    4           2.250000   
                 Transaction Fees                   4           2.250000   
Finance          Software                           4           2.250000   
Operations       Transaction Fees                   4           2.250000   

                                        max_anomaly_score  
department       category                                  
Finance          Professional Services                  3  
                 Transaction Fees                       2  
HR               Recruitment                            4  
Customer Support Outsourcing                            3  
Sales            Software                               3  
Finance          Subscription                           2  
IT               Subscription                           2  
Sales            Travel                                 3  
Finance          Audit                                  3  
HR               Service Revenue                        3  
IT               Software                               3  
Customer Support Software                               3  
Finance          Bank Charges                           3  
HR               Benefits                               3  
                 Training                               3  
Operations       Logistics                              3  
Customer Support Service Revenue                        3  
                 Transaction Fees                       3  
Finance          Software                               3  
Operations       Transaction Fees                       3

In [10]:
# Step 10 — Validate anomaly detection against ground truth

ground_truth_df = pd.read_csv("../dataset/ground_truth.csv")

print("Ground truth loaded successfully!")
print("Shape:", ground_truth_df.shape)

display(ground_truth_df.head(10))

print("\nGround truth columns:")
print(ground_truth_df.columns.tolist())

Ground truth loaded successfully!
Shape: (5, 6)


,scenario_id,period,metric,expected_root_cause,expected_vendor,description
0,SCN001,2026-04,Operating Expense,Marketing > Advertising,AdSphere / MarketBoost / SearchPro,Marketing Advertising spending was intentional...
1,SCN002,2026-05,Revenue,South Region > Revenue,Customer segment,South-region revenue transaction values were i...
2,SCN003,2026-03,Operating Expense,IT > Software > CloudSuite,CloudSuite,CloudSuite software spending was intentionally...
3,SCN004,2026-02,Operating Expense,Operations > Logistics,FastFreight / MoveRight / ShipNow,Operations logistics spending was intentionall...
4,SCN005,2026-06,Operating Expense,HR > Recruitment,TalentBridge / HireFast,HR recruitment spending was intentionally incr...



Ground truth columns:
['scenario_id', 'period', 'metric', 'expected_root_cause', 'expected_vendor', 'description']


In [11]:
# Step 10B — Match detected anomalies against ground truth

validation_results = []

for _, gt in ground_truth_df.iterrows():

    root_parts = gt["expected_root_cause"].split(" > ")

    expected_department = root_parts[0]
    expected_category = root_parts[1]

    matches = anomaly_df[
        (anomaly_df["period"] == gt["period"]) &
        (anomaly_df["department"] == expected_department) &
        (anomaly_df["category"] == expected_category)
    ]

    if len(matches) > 0:
        detected_as_anomaly = int(matches["is_anomaly"].max())
        max_anomaly_score = matches["anomaly_score"].max()
    else:
        detected_as_anomaly = 0
        max_anomaly_score = 0

    validation_results.append({
        "scenario_id": gt["scenario_id"],
        "period": gt["period"],
        "expected_department": expected_department,
        "expected_category": expected_category,
        "detected_as_anomaly": detected_as_anomaly,
        "max_anomaly_score": max_anomaly_score
    })


validation_results_df = pd.DataFrame(validation_results)

print("Ground-truth validation results:")
display(validation_results_df)

detected_count = validation_results_df["detected_as_anomaly"].sum()
total_scenarios = len(validation_results_df)

print(f"\nKnown scenarios detected: {detected_count} out of {total_scenarios}")

print(
    "Detection rate:",
    round(detected_count / total_scenarios * 100, 2),
    "%"
)

Ground-truth validation results:


,scenario_id,period,expected_department,expected_category,detected_as_anomaly,max_anomaly_score
0,SCN001,2026-04,Marketing,Advertising,1,1
1,SCN002,2026-05,South Region,Revenue,0,0
2,SCN003,2026-03,IT,Software,1,2
3,SCN004,2026-02,Operations,Logistics,1,3
4,SCN005,2026-06,HR,Recruitment,1,2



Known scenarios detected: 4 out of 5
Detection rate: 80.0 %


In [12]:
# Step 11 — Investigate missed ground-truth scenarios

missed_scenarios = validation_results_df[
    validation_results_df["detected_as_anomaly"] == 0
]

print("Missed ground-truth scenarios:")
display(missed_scenarios)


for _, scenario in missed_scenarios.iterrows():

    print("\n" + "=" * 70)
    print(f"Scenario: {scenario['scenario_id']}")
    print(f"Period: {scenario['period']}")
    print(f"Expected Department: {scenario['expected_department']}")
    print(f"Expected Category: {scenario['expected_category']}")

    matching_rows = anomaly_df[
        (anomaly_df["period"] == scenario["period"]) &
        (anomaly_df["department"] == scenario["expected_department"]) &
        (anomaly_df["category"] == scenario["expected_category"])
    ]

    if len(matching_rows) == 0:
        print("No matching department-category row found.")
    else:
        print("Matching row found:")
        
        display(
            matching_rows[
                [
                    "period",
                    "department",
                    "category",
                    "anomaly_score",
                    "is_anomaly",
                    "anomaly_signals"
                ]
            ]
        )

Missed ground-truth scenarios:


,scenario_id,period,expected_department,expected_category,detected_as_anomaly,max_anomaly_score
1,SCN002,2026-05,South Region,Revenue,0,0



Scenario: SCN002
Period: 2026-05
Expected Department: South Region
Expected Category: Revenue
No matching department-category row found.


In [13]:
# Step 12 — Diagnose ground-truth alignment and anomaly thresholds

print("=== Ground-truth periods available in feature dataset ===")

gt_periods = ground_truth_df["period"].unique()

for period in gt_periods:
    print(f"\nPeriod: {period}")

    period_rows = anomaly_df[
        anomaly_df["period"] == period
    ]

    print("Departments:")
    print(sorted(period_rows["department"].dropna().unique().tolist()))

    print("\nRelevant anomaly rows:")
    display(
        period_rows[
            [
                "period",
                "department",
                "category",
                "total_amount",
                "transaction_count",
                "spending_deviation_pct",
                "transaction_count_deviation_pct",
                "anomaly_score",
                "is_anomaly",
                "anomaly_signals"
            ]
        ].sort_values("anomaly_score", ascending=False).head(15)
    )


print("\n" + "=" * 70)
print("SCN001 — Marketing / Advertising")

scn001 = anomaly_df[
    (anomaly_df["period"] == "2026-04") &
    (anomaly_df["department"] == "Marketing") &
    (anomaly_df["category"] == "Advertising")
]

display(scn001)


print("\n" + "=" * 70)
print("SCN002 — South Region / Revenue")

scn002_period = anomaly_df[
    anomaly_df["period"] == "2026-05"
]

print("Departments available in 2026-05:")
print(
    sorted(
        scn002_period["department"]
        .dropna()
        .unique()
        .tolist()
    )
)

print("\nRows containing 'Revenue' in 2026-05:")
display(
    scn002_period[
        scn002_period["category"]
        .astype(str)
        .str.contains("Revenue", case=False, na=False)
    ][
        [
            "period",
            "department",
            "category",
            "total_amount",
            "anomaly_score",
            "is_anomaly",
            "anomaly_signals"
        ]
    ]
)

=== Ground-truth periods available in feature dataset ===

Period: 2026-04
Departments:
['Customer Support', 'Finance', 'HR', 'IT', 'Marketing', 'Operations', 'Sales']

Relevant anomaly rows:


,period,department,category,total_amount,transaction_count,spending_deviation_pct,transaction_count_deviation_pct,anomaly_score,is_anomaly,anomaly_signals
851,2026-04,Finance,Professional Services,571050.75,66,57.670849,42.446043,3,1,"high budget utilization, high spending deviati..."
866,2026-04,IT,Hardware,1004117.38,56,-21.635151,-24.153499,3,1,"high budget utilization, high spending deviati..."
852,2026-04,Finance,Service Revenue,509550.43,19,-30.321765,-39.682540,2,1,"high spending deviation, high transaction devi..."
847,2026-04,Customer Support,Transaction Fees,777430.52,38,-27.282166,-20.649652,2,1,"high spending deviation, high transaction devi..."
856,2026-04,HR,Benefits,349589.99,48,35.114506,5.494505,2,1,"high budget utilization, high spending deviation"
870,2026-04,IT,Subscription,1450774.50,62,47.580695,38.633540,2,1,"high spending deviation, high transaction devi..."
871,2026-04,IT,Transaction Fees,1444510.25,61,53.368841,41.494845,2,1,"high spending deviation, high transaction devi..."
858,2026-04,HR,Recruitment,305814.87,51,26.811847,12.915129,2,1,"high budget utilization, high spending deviation"
888,2026-04,Sales,Client Entertainment,918296.73,90,22.101130,8.000000,2,1,"high budget utilization, high spending deviation"
859,2026-04,HR,Service Revenue,915107.97,36,33.798782,18.248175,1,0,high spending deviation



Period: 2026-05
Departments:
['Customer Support', 'Finance', 'HR', 'IT', 'Marketing', 'Operations', 'Sales']

Relevant anomaly rows:


,period,department,category,total_amount,transaction_count,spending_deviation_pct,transaction_count_deviation_pct,anomaly_score,is_anomaly,anomaly_signals
899,2026-05,Customer Support,Software,1088637.53,101,53.611926,34.766494,3,1,"high budget utilization, high spending deviati..."
902,2026-05,Customer Support,Training,729236.71,95,33.466054,24.544792,3,1,"high budget utilization, high spending deviati..."
901,2026-05,Customer Support,Support Tools,939992.79,88,24.553989,9.771310,2,1,"high budget utilization, high spending deviation"
898,2026-05,Customer Support,Service Revenue,794257.59,35,-27.288253,-25.619835,2,1,"high spending deviation, high transaction devi..."
910,2026-05,Finance,Subscription,832639.58,43,23.712152,39.963834,2,1,"high spending deviation, high transaction devi..."
907,2026-05,Finance,Professional Services,243196.34,34,-32.851899,-26.618705,2,1,"high spending deviation, high transaction devi..."
923,2026-05,IT,Product Sales,652256.58,32,-36.605130,-29.411765,2,1,"high spending deviation, high transaction devi..."
920,2026-05,IT,Cloud Services,1932156.48,98,50.598756,36.532508,2,1,"high spending deviation, high transaction devi..."
938,2026-05,Operations,Product Sales,838168.04,44,-37.278343,-26.394052,2,1,"high spending deviation, high transaction devi..."
924,2026-05,IT,Service Revenue,562873.75,30,-46.044701,-34.386391,2,1,"high spending deviation, high transaction devi..."



Period: 2026-03
Departments:
['Customer Support', 'Finance', 'HR', 'IT', 'Marketing', 'Operations', 'Sales']

Relevant anomaly rows:


...

,period,department,category,total_amount,transaction_count,spending_deviation_pct,transaction_count_deviation_pct,anomaly_score,is_anomaly,anomaly_signals
965,2026-06,Finance,Software,485642.40,71,45.399473,38.611714,3,1,"high budget utilization, high spending deviati..."
974,2026-06,HR,Training,166826.75,37,-21.414695,-22.014052,3,1,"high budget utilization, high spending deviati..."
960,2026-06,Finance,Audit,493973.46,65,37.328014,26.349892,3,1,"high budget utilization, high spending deviati..."
966,2026-06,Finance,Subscription,851136.67,39,26.460417,26.943942,2,1,"high spending deviation, high transaction devi..."
970,2026-06,HR,Recruitment,481956.36,50,99.852205,10.701107,2,1,"high budget utilization, high spending deviation"
989,2026-06,Marketing,Software,876269.84,71,-14.456000,-22.498484,2,1,"high budget utilization, high transaction devi..."
987,2026-06,Marketing,Product Sales,720213.00,38,-37.524570,-29.918033,2,1,"high spending deviation, high transaction devi..."
979,2026-06,IT,Product Sales,1305396.39,55,26.875583,21.323529,2,1,"high spending deviation, high transaction devi..."
1004,2026-06,Sales,Software,617199.52,70,-26.142793,-17.214192,2,1,"high budget utilization, high spending deviation"
1005,2026-06,Sales,Subscription,794303.67,35,-27.304054,-29.922136,2,1,"high spending deviation, high transaction devi..."



SCN001 — Marketing / Advertising


,period,department,category,transaction_count,total_amount,budget,variance,budget_utilization_pct,variance_pct,budget_available,...,rolling_3m_avg_spending,rolling_spending_deviation_pct,high_budget_utilization,high_spending_deviation,high_transaction_deviation,large_period_change,large_rolling_deviation,anomaly_score,is_anomaly,anomaly_signals
872,2026-04,Marketing,Advertising,77,2654035.5,2682854.36,-28818.86,98.925813,-1.074187,1,...,5.465376e+07,1.190604,False,True,False,False,False,1,1,high spending deviation



SCN002 — South Region / Revenue
Departments available in 2026-05:
['Customer Support', 'Finance', 'HR', 'IT', 'Marketing', 'Operations', 'Sales']

Rows containing 'Revenue' in 2026-05:


,period,department,category,total_amount,anomaly_score,is_anomaly,anomaly_signals
898,2026-05,Customer Support,Service Revenue,794257.59,2,1,"high spending deviation, high transaction devi..."
908,2026-05,Finance,Service Revenue,615042.94,0,0,
915,2026-05,HR,Service Revenue,677828.54,0,0,
924,2026-05,IT,Service Revenue,562873.75,2,1,"high spending deviation, high transaction devi..."
932,2026-05,Marketing,Service Revenue,931887.87,1,0,high spending deviation
939,2026-05,Operations,Service Revenue,1000837.01,0,0,
947,2026-05,Sales,Service Revenue,1174007.92,1,0,high transaction deviation


In [14]:
# Step 13 — Diagnose detection quality and ground-truth consistency

print("=" * 70)
print("SCN001 — Detection diagnosis")
print("=" * 70)

scn001 = anomaly_df[
    (anomaly_df["period"] == "2026-04") &
    (anomaly_df["department"] == "Marketing") &
    (anomaly_df["category"] == "Advertising")
].copy()

display(
    scn001[
        [
            "period",
            "department",
            "category",
            "total_amount",
            "budget",
            "budget_utilization_pct",
            "variance",
            "variance_pct",
            "spending_deviation_pct",
            "transaction_count_deviation_pct",
            "anomaly_score",
            "is_anomaly",
            "anomaly_signals"
        ]
    ]
)


print("\n" + "=" * 70)
print("SCN002 — Ground-truth schema diagnosis")
print("=" * 70)

print("\nExpected:")
print("Department = South Region")
print("Category   = Revenue")
print("Period     = 2026-05")

print("\nActual departments in dataset:")

actual_departments = sorted(
    anomaly_df[
        anomaly_df["period"] == "2026-05"
    ]["department"].dropna().unique().tolist()
)

print(actual_departments)


print("\nActual Revenue-related rows:")

revenue_rows = anomaly_df[
    (anomaly_df["period"] == "2026-05") &
    (
        anomaly_df["category"]
        .astype(str)
        .str.contains("Revenue", case=False, na=False)
    )
]

display(
    revenue_rows[
        [
            "period",
            "department",
            "category",
            "total_amount",
            "spending_deviation_pct",
            "transaction_count_deviation_pct",
            "anomaly_score",
            "is_anomaly",
            "anomaly_signals"
        ]
    ]
)


print("\n" + "=" * 70)
print("Ground-truth consistency check")
print("=" * 70)

# IMPORTANT:
# validation_results_df already contains the parsed
# expected_department and expected_category values.

print("\nExpected departments from ground truth:")
print(
    sorted(
        validation_results_df[
            "expected_department"
        ].dropna().unique().tolist()
    )
)

print("\nExpected categories from ground truth:")
print(
    sorted(
        validation_results_df[
            "expected_category"
        ].dropna().unique().tolist()
    )
)

print("\nActual departments in anomaly dataset:")
print(
    sorted(
        anomaly_df["department"]
        .dropna()
        .unique()
        .tolist()
    )
)

print("\nGround-truth departments missing from dataset:")

gt_departments = set(
    validation_results_df["expected_department"].dropna()
)

actual_departments = set(
    anomaly_df["department"].dropna()
)

missing_departments = gt_departments - actual_departments

print(sorted(missing_departments))

SCN001 — Detection diagnosis


,period,department,category,total_amount,budget,budget_utilization_pct,variance,variance_pct,spending_deviation_pct,transaction_count_deviation_pct,anomaly_score,is_anomaly,anomaly_signals
872,2026-04,Marketing,Advertising,2654035.5,2682854.36,98.925813,-28818.86,-1.074187,77.841508,-11.719745,1,1,high spending deviation



SCN002 — Ground-truth schema diagnosis

Expected:
Department = South Region
Category   = Revenue
Period     = 2026-05

Actual departments in dataset:
['Customer Support', 'Finance', 'HR', 'IT', 'Marketing', 'Operations', 'Sales']

Actual Revenue-related rows:


,period,department,category,total_amount,spending_deviation_pct,transaction_count_deviation_pct,anomaly_score,is_anomaly,anomaly_signals
898,2026-05,Customer Support,Service Revenue,794257.59,-27.288253,-25.619835,2,1,"high spending deviation, high transaction devi..."
908,2026-05,Finance,Service Revenue,615042.94,-15.896241,1.587302,0,0,
915,2026-05,HR,Service Revenue,677828.54,-0.894062,5.109489,0,0,
924,2026-05,IT,Service Revenue,562873.75,-46.044701,-34.386391,2,1,"high spending deviation, high transaction devi..."
932,2026-05,Marketing,Service Revenue,931887.87,-21.374215,-19.018405,1,0,high spending deviation
939,2026-05,Operations,Service Revenue,1000837.01,-19.908385,-10.176125,0,0,
947,2026-05,Sales,Service Revenue,1174007.92,13.094366,20.415225,1,0,high transaction deviation



Ground-truth consistency check

Expected departments from ground truth:
['HR', 'IT', 'Marketing', 'Operations', 'South Region']

Expected categories from ground truth:
['Advertising', 'Logistics', 'Recruitment', 'Revenue', 'Software']

Actual departments in anomaly dataset:
['Customer Support', 'Finance', 'HR', 'IT', 'Marketing', 'Operations', 'Sales']

Ground-truth departments missing from dataset:
['South Region']


In [15]:
print("Anomaly dataset columns:")
print(anomaly_df.columns.tolist())

print("\nColumns containing region/location/segment information:")

region_cols = [
    col for col in anomaly_df.columns
    if any(
        word in col.lower()
        for word in ["region", "location", "area", "zone", "segment"]
    )
]

print(region_cols)

Anomaly dataset columns:
['period', 'department', 'category', 'transaction_count', 'total_amount', 'budget', 'variance', 'budget_utilization_pct', 'variance_pct', 'budget_available', 'avg_transaction_value', 'spending_deviation_pct', 'transaction_count_deviation_pct', 'period_total_spending', 'period_spending_change_pct', 'rolling_3m_avg_spending', 'rolling_spending_deviation_pct', 'high_budget_utilization', 'high_spending_deviation', 'high_transaction_deviation', 'large_period_change', 'large_rolling_deviation', 'anomaly_score', 'is_anomaly', 'anomaly_signals']

Columns containing region/location/segment information:
[]


In [16]:
print("Ground-truth dataset columns:")
print(ground_truth_df.columns.tolist())

print("\nGround-truth data:")
display(ground_truth_df)

print("\nAnomaly dataset categories:")
print(sorted(anomaly_df["category"].dropna().unique()))

print("\nAnomaly dataset departments:")
print(sorted(anomaly_df["department"].dropna().unique()))

Ground-truth dataset columns:
['scenario_id', 'period', 'metric', 'expected_root_cause', 'expected_vendor', 'description']

Ground-truth data:


,scenario_id,period,metric,expected_root_cause,expected_vendor,description
0,SCN001,2026-04,Operating Expense,Marketing > Advertising,AdSphere / MarketBoost / SearchPro,Marketing Advertising spending was intentional...
1,SCN002,2026-05,Revenue,South Region > Revenue,Customer segment,South-region revenue transaction values were i...
2,SCN003,2026-03,Operating Expense,IT > Software > CloudSuite,CloudSuite,CloudSuite software spending was intentionally...
3,SCN004,2026-02,Operating Expense,Operations > Logistics,FastFreight / MoveRight / ShipNow,Operations logistics spending was intentionall...
4,SCN005,2026-06,Operating Expense,HR > Recruitment,TalentBridge / HireFast,HR recruitment spending was intentionally incr...



Anomaly dataset categories:
['Advertising', 'Audit', 'Bank Charges', 'Benefits', 'Client Entertainment', 'Cloud Services', 'Commissions', 'Content', 'Cybersecurity', 'Events', 'Hardware', 'Logistics', 'Maintenance', 'Outsourcing', 'Product Sales', 'Professional Services', 'Recruitment', 'Service Revenue', 'Software', 'Subscription', 'Supplies', 'Support Tools', 'Training', 'Transaction Fees', 'Travel', 'Utilities']

Anomaly dataset departments:
['Customer Support', 'Finance', 'HR', 'IT', 'Marketing', 'Operations', 'Sales']


In [17]:
print("Revenue-related categories:")
print(
    anomaly_df[
        anomaly_df["category"].astype(str).str.contains(
            "revenue", case=False, na=False
        )
    ][
        ["period", "department", "category",
         "total_amount", "anomaly_score", "is_anomaly"]
    ].to_string(index=False)
)

Revenue-related categories:
 period       department        category  total_amount  anomaly_score  is_anomaly
2025-01 Customer Support Service Revenue    1125824.05              0           0
2025-01          Finance Service Revenue     588653.71              0           0
2025-01               HR Service Revenue     528749.35              2           1
2025-01               IT Service Revenue     951131.27              0           0
2025-01        Marketing Service Revenue    1274500.26              0           0
2025-01       Operations Service Revenue    1259244.31              0           0
2025-01            Sales Service Revenue    1064062.99              0           0
2025-02 Customer Support Service Revenue     792629.72              2           1
2025-02          Finance Service Revenue     766667.63              0           0
2025-02               HR Service Revenue     659197.01              0           0
2025-02               IT Service Revenue     841494.56              1 

In [18]:
print("Unique categories containing revenue:")
print(
    sorted(
        anomaly_df.loc[
            anomaly_df["category"].astype(str).str.contains(
                "revenue", case=False, na=False
            ),
            "category"
        ].dropna().unique()
    )
)

Unique categories containing revenue:
['Service Revenue']


In [19]:
# Step 13B — Validate ground-truth scenarios with schema awareness

results = []

for _, gt in ground_truth_df.iterrows():

    scenario_id = gt["scenario_id"]
    period = gt["period"]
    root_cause = str(gt["expected_root_cause"])

    # Extract hierarchy
    parts = [p.strip() for p in root_cause.split(">")]

    expected_department = parts[0]
    expected_category = parts[1] if len(parts) > 1 else None
    expected_detail = parts[2] if len(parts) > 2 else None

    # SCN002: region/segment information is absent
    if expected_department == "South Region":
        results.append({
            "scenario_id": scenario_id,
            "period": period,
            "expected_root_cause": root_cause,
            "status": "UNVERIFIABLE",
            "reason": "Region/segment field is not available in anomaly dataset"
        })
        continue

    # Normal department + category matching
    mask = (
        (anomaly_df["period"].astype(str) == str(period)) &
        (anomaly_df["department"].astype(str) == expected_department)
    )

    # Revenue alias
    if expected_category == "Revenue":
        mask &= anomaly_df["category"].astype(str).eq("Service Revenue")
    else:
        mask &= anomaly_df["category"].astype(str).eq(expected_category)

    matching_rows = anomaly_df[mask]

    detected = int(matching_rows["is_anomaly"].max()) if len(matching_rows) > 0 else 0

    results.append({
        "scenario_id": scenario_id,
        "period": period,
        "expected_root_cause": root_cause,
        "status": "DETECTED" if detected == 1 else "MISSED",
        "reason": (
            f"{len(matching_rows)} matching row(s)"
            if len(matching_rows) > 0
            else "No matching row found"
        )
    })

validation_final_df = pd.DataFrame(results)

display(validation_final_df)

,scenario_id,period,expected_root_cause,status,reason
0,SCN001,2026-04,Marketing > Advertising,DETECTED,1 matching row(s)
1,SCN002,2026-05,South Region > Revenue,UNVERIFIABLE,Region/segment field is not available in anoma...
2,SCN003,2026-03,IT > Software > CloudSuite,DETECTED,1 matching row(s)
3,SCN004,2026-02,Operations > Logistics,DETECTED,1 matching row(s)
4,SCN005,2026-06,HR > Recruitment,DETECTED,1 matching row(s)


In [24]:
# Step 14 — Diagnose SCN001 anomaly score

scn001 = anomaly_df[
    (anomaly_df["period"].astype(str) == "2026-04") &
    (anomaly_df["department"] == "Marketing") &
    (anomaly_df["category"] == "Advertising")
]

print("SCN001 row:")
display(scn001.T)

SCN001 row:


,872
period,2026-04
department,Marketing
category,Advertising
transaction_count,77
total_amount,2654035.5
budget,2682854.36
variance,-28818.86
budget_utilization_pct,98.925813
variance_pct,-1.074187
budget_available,1


In [28]:
# Compare SCN001 against the anomaly-score distribution

print("Anomaly score distribution:")
print(anomaly_df["anomaly_score"].value_counts().sort_index())

print("\nSCN001:")
print(
    scn001[
        [
            "period",
            "department",
            "category",
            "total_amount",
            "budget",
            "budget_utilization_pct",
            "variance",
            "variance_pct",
            "spending_deviation_pct",
            "transaction_count",
            "transaction_count_deviation_pct",
            "anomaly_score",
            "is_anomaly",
            "anomaly_signals"
        ]
    ].to_string(index=False)
)

Anomaly score distribution:
anomaly_score
0    420
1    398
2    148
3     41
4      1
Name: count, dtype: int64

SCN001:
 period department    category  total_amount     budget  budget_utilization_pct  variance  variance_pct  spending_deviation_pct  transaction_count  transaction_count_deviation_pct  anomaly_score  is_anomaly         anomaly_signals
2026-04  Marketing Advertising     2654035.5 2682854.36               98.925813 -28818.86     -1.074187               77.841508                 77                       -11.719745              1           1 high spending deviation


In [30]:
# Diagnose score-1 rows by anomaly signal

score_1 = anomaly_df[anomaly_df["anomaly_score"] == 1].copy()

print("Total score-1 rows:", len(score_1))

print("\nScore-1 anomaly signals:")
print(score_1["anomaly_signals"].value_counts())

print("\nScore-1 rows with high spending deviation:")
print(
    score_1["high_spending_deviation"]
    .value_counts(dropna=False)
)

print("\nScore-1 spending deviation statistics:")
print(
    score_1["spending_deviation_pct"].describe()
)

print("\nSCN001:")
print(
    anomaly_df[
        (anomaly_df["period"] == "2026-04") &
        (anomaly_df["department"] == "Marketing") &
        (anomaly_df["category"] == "Advertising")
    ][[
        "period",
        "department",
        "category",
        "spending_deviation_pct",
        "high_spending_deviation",
        "anomaly_score",
        "is_anomaly",
        "anomaly_signals"
    ]]
)

Total score-1 rows: 398

Score-1 anomaly signals:
anomaly_signals
high budget utilization       255
high spending deviation        82
high transaction deviation     42
large period change            19
Name: count, dtype: int64

Score-1 rows with high spending deviation:
high_spending_deviation
False    316
True      82
Name: count, dtype: int64

Score-1 spending deviation statistics:
count    398.000000
mean      -0.628481
std       15.871015
min      -36.629237
25%      -12.441640
50%       -1.737945
75%       10.801169
max       77.841508
Name: spending_deviation_pct, dtype: float64

SCN001:
      period department     category  spending_deviation_pct  \
872  2026-04  Marketing  Advertising               77.841508   

     high_spending_deviation  anomaly_score  is_anomaly  \
872                     True              1           1   

             anomaly_signals  
872  high spending deviation  


In [23]:
# Check score-1 rows with high spending deviation

high_spend_score1 = anomaly_df[
    (anomaly_df["anomaly_score"] == 1) &
    (anomaly_df["high_spending_deviation"] == True)
].copy()

print("Score-1 + high spending deviation:", len(high_spend_score1))

print("\nSpending deviation distribution:")
print(
    high_spend_score1["spending_deviation_pct"]
    .describe()
)

print("\nRows with spending deviation >= 50%:")
print(
    high_spend_score1[
        high_spend_score1["spending_deviation_pct"] >= 50
    ][[
        "period",
        "department",
        "category",
        "spending_deviation_pct",
        "transaction_count_deviation_pct",
        "anomaly_score",
        "is_anomaly",
        "anomaly_signals"
    ]]
)

Score-1 + high spending deviation: 82

Spending deviation distribution:
count    82.000000
mean      0.828138
std      27.978328
min     -36.629237
25%     -25.402153
50%      20.014748
75%      24.249428
max      77.841508
Name: spending_deviation_pct, dtype: float64

Rows with spending deviation >= 50%:
      period department     category  spending_deviation_pct  \
872  2026-04  Marketing  Advertising               77.841508   

     transaction_count_deviation_pct  anomaly_score  is_anomaly  \
872                       -11.719745              1           1   

             anomaly_signals  
872  high spending deviation  


In [31]:
# Step 15 — Cash-flow forecasting setup

# Aggregate monthly spending
cashflow_df = (
    anomaly_df
    .groupby("period", as_index=False)
    .agg(
        total_spending=("total_amount", "sum")
    )
)

# Ensure chronological order
cashflow_df["period"] = pd.to_datetime(
    cashflow_df["period"].astype(str)
)

cashflow_df = cashflow_df.sort_values("period").reset_index(drop=True)

# Net change in spending
cashflow_df["spending_change"] = (
    cashflow_df["total_spending"].diff()
)

display(cashflow_df)

,period,total_spending,spending_change
0,2025-01-01,53363242.28,NaN
1,2025-02-01,49205537.23,-4157705.05
2,2025-03-01,59577029.34,10371492.11
3,2025-04-01,54481805.98,-5095223.36
4,2025-05-01,58516281.48,4034475.50
5,2025-06-01,51348851.16,-7167430.32
6,2025-07-01,51614641.62,265790.46
7,2025-08-01,50829734.44,-784907.18
8,2025-09-01,46872855.13,-3956879.31
9,2025-10-01,48992050.97,2119195.84


In [32]:
# Step 16 — Simple next-month cash-flow forecast

# Use the average of the last 3 months' spending changes
recent_changes = cashflow_df["spending_change"].tail(3)

avg_change = recent_changes.mean()

last_spending = cashflow_df["total_spending"].iloc[-1]

forecast_next_month = last_spending + avg_change

print(f"Last month spending: ₹{last_spending:,.2f}")
print(f"Average recent change: ₹{avg_change:,.2f}")
print(f"Forecast next month spending: ₹{forecast_next_month:,.2f}")

Last month spending: ₹52,598,244.30
Average recent change: ₹-1,387,220.65
Forecast next month spending: ₹51,211,023.65


In [33]:
# Step 17 — Risk assessment and recommendation

# Compare forecast with last month's spending
forecast_change_pct = (
    (forecast_next_month - last_spending)
    / last_spending
) * 100

if forecast_change_pct <= -10:
    risk_level = "HIGH"
    recommendation = "Spending is expected to decline significantly. Review major expense categories and maintain sufficient cash reserves."
elif forecast_change_pct <= -5:
    risk_level = "MEDIUM"
    recommendation = "Spending is expected to decline moderately. Monitor upcoming expenses and cash requirements."
else:
    risk_level = "LOW"
    recommendation = "Spending is relatively stable. Continue monitoring anomalies and budget utilization."

print(f"Forecast change: {forecast_change_pct:.2f}%")
print(f"Risk Level: {risk_level}")
print(f"Recommendation: {recommendation}")

Forecast change: -2.64%
Risk Level: LOW
Recommendation: Spending is relatively stable. Continue monitoring anomalies and budget utilization.


In [34]:
# Step 18 — Final FinTrace AI summary

print("=" * 55)
print("           FINTRACE AI — FINANCIAL RISK SUMMARY")
print("=" * 55)

print(f"Latest Month Spending : ₹{last_spending:,.2f}")
print(f"Forecast Next Month   : ₹{forecast_next_month:,.2f}")
print(f"Forecast Change       : {forecast_change_pct:.2f}%")
print(f"Overall Risk Level    : {risk_level}")

print("\nRecommendation:")
print(recommendation)

print("\nAnomalies Detected:")
print(int(anomaly_df["is_anomaly"].sum()))

print("=" * 55)

           FINTRACE AI — FINANCIAL RISK SUMMARY
Latest Month Spending : ₹52,598,244.30
Forecast Next Month   : ₹51,211,023.65
Forecast Change       : -2.64%
Overall Risk Level    : LOW

Recommendation:
Spending is relatively stable. Continue monitoring anomalies and budget utilization.

Anomalies Detected:
191


In [35]:
# Step 19 — Save final anomaly results

output_path = "../dataset/anomaly_results.csv"

anomaly_df.to_csv(output_path, index=False)

print(f"Saved final anomaly results to: {output_path}")
print(f"Rows: {len(anomaly_df):,}")
print(f"Columns: {len(anomaly_df.columns):,}")

Saved final anomaly results to: ../dataset/anomaly_results.csv
Rows: 1,008
Columns: 25
